In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split
from surprise import accuracy

ModuleNotFoundError: No module named 'surprise'

In [ ]:
PROCESSED = Path("../Dataset/Processed")
raw_dir = Path("../Dataset/Raw")

df_reviews = pd.read_csv(PROCESSED / "reviews_with_sentiment.csv", low_memory=False)
df_ratings = df_reviews[["user", "ID", "rating"]].dropna()
df_ratings = df_ratings.assign(rating=df_ratings["rating"].astype(float))
df_ratings.head()

In [ ]:
reader = Reader(rating_scale=(df_ratings.rating.min(), df_ratings.rating.max()))
data = Dataset.load_from_df(df_ratings[["user","ID","rating"]], reader)

trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

algo = SVD(n_factors=100, n_epochs=20, lr_all=0.005, reg_all=0.02, random_state=42)
algo.fit(trainset)

predictions = algo.test(testset)
print("CF RMSE:", accuracy.rmse(predictions))
print("CF MAE:", accuracy.mae(predictions))

In [ ]:
all_items = df_ratings["ID"].unique().tolist()

def cf_scores_for_seed(seed_game_ids, user_id="__temp_user__"):
    global_mean = df_ratings["rating"].mean()
    seed_rating_map = {}
    for gid in seed_game_ids:
        seed_vals = df_ratings.loc[df_ratings["ID"]==gid, "rating"]
        seed_rating_map[gid] = seed_vals.mean() if len(seed_vals)>0 else global_mean

    sample_users = df_ratings[df_ratings["ID"].isin(seed_game_ids)]["user"].unique()
    if len(sample_users) > 5000:
        sample_users = np.random.choice(sample_users, 5000, replace=False)
    scores = {}
    for item in all_items:
        preds = []
        for u in sample_users[:200]:
            p = algo.predict(uid=u, iid=item).est
            preds.append(p)
        scores[item] = np.mean(preds) if preds else global_mean
    return pd.Series(scores)

In [ ]:
import joblib
joblib.dump(algo, "../Dataset/Processed/svd_cf_model.joblib")